In [2]:
#Importando Bibliotecas que estou utilizando para a análise
import pandas as pd
from ydata_profiling import ProfileReport
import os

In [3]:
#Caminho da Base de dados
caminho_dados = r"C:\Users\gabri\OneDrive\Belgeler\GitHub\Python\AnalisesRedebras\Referência\Dados\RelPedidoCli.xls"
df_comercial = pd.read_excel(caminho_dados)

# Sobrescreve o dataframe removendo a última linha (Linha total que o próprio ERP ja envia junto)
df_comercial = df_comercial.iloc[:-1]

In [ ]:
## ALTERAÇÕES DE LIMPEZA E FORMATAÇÃO ##

#Modificando Número para Texto (Pedidos e N.F -> Precisam ser identificados como texto para a análise)
colunas_texto = ['PEDIDO', 'NF']
for col in colunas_texto:
    df_comercial[col] = (
        df_comercial[col].astype(str)
    )

#------------------------------------------------------------------------------#
# Forçamos as colunas a serem strings para limpeza manual 
# Formatando os Valores de Preços e Custos
colunas_preco = ['TOTAL CUSTO', 'TOTAL ITENS', 'TOTAL PEDIDO', 'QTDE VENDA']
for col in colunas_preco:
    df_comercial[col] = (
        df_comercial[col].astype(str)
    )

for col in colunas_preco:
    # 1. Troca a vírgula pelo ponto decimal (1200,50 -> 1200.50)
    df_comercial[col] = df_comercial[col].str.replace(',', '.', regex=False)
    # 2. Converte para número real (float)
    df_comercial[col] = pd.to_numeric(df_comercial[col], errors='coerce')
    # 3. Arredonda para 2 casas decimais
    df_comercial[colunas_preco] = df_comercial[colunas_preco].round(2)

#------------------------------------------------------------------------------#
#Formatando Colunas de Datas
# Modificando as colunas de Data para o tipo Data
colunas_data = ['DATA', 'NF. EMISSAO']

for col in colunas_data:
    df_comercial[col] = (
    pd.to_datetime(df_comercial[col], dayfirst=True, errors='coerce')
    )

#------------------------------------------------------------------------------#
#Limpeza de Colunas não necessárias para o projeto (Deletando Colunas)
colunas_desnecessarias = ['EMP', 'REPRESENTANTE', 'TIPO', 'INTEGRAÇÃO', 'OUTRAS DESP', 'VLR. FRETE', 'DT. RETIRADA', 'FRETE TYPE', 'FIN',
                           'TOTAL ITENS C/ IPI', 'TOTAL ITENS', 'MKP']

df_comercial = df_comercial.drop(columns=colunas_desnecessarias, errors='ignore')   

#------------------------------------------------------------------------------#
#Ajustando a coluna de %MKP 
# 1. Garantimos que é string, removemos '%' e espaços
df_comercial['MKP'] = df_comercial['MKP'].astype(str).str.replace('%', '').str.strip()

# 2. Trocamos a vírgula pelo ponto (padrão brasileiro -> americano)
df_comercial['MKP'] = df_comercial['MKP'].str.replace(',', '.')

# 3. Convertemos para número
df_comercial['MKP'] = pd.to_numeric(df_comercial['MKP'], errors='coerce')

# 4. Arredondamos para evitar dízimas (como vimos nas outras colunas)
df_comercial['MKP'] = df_comercial['MKP'].round(2)

#------------------------------------------------------------------------------#
# Criação de colunas
# Lucro R$ (Para saber o lucro de cada venda em dinheiro)
df_comercial['LUCRO R$'] = (
    df_comercial['TOTAL PEDIDO'] - df_comercial['TOTAL CUSTO']
)
df_comercial['LUCRO R$'] = df_comercial['LUCRO R$'].round(2)


df_comercial['TOTAL VENDAS MÊS'] = df_comercial['TOTAL PEDIDO'].sum()

# Formatando a coluna de Data para o formato brasileiro (dd/mm/yyyy)
df_comercial['DATA'] = df_comercial['DATA'].dt.strftime('%d/%m/%Y')

# Formatando a coluna de Data NF. EMISSAO para o formato brasileiro (dd/mm/yyyy)
df_comercial['NF. EMISSAO'] = df_comercial['NF. EMISSAO'].dt.strftime('%d/%m/%Y')

##--------------------------------------##
# Calculando o MKP REAL de cada venda
# 1.Markup com o multiplicador real (Preço de Venda / Custo) para comparar com o MKP do ERP
df_comercial['MKP REAL'] = df_comercial['TOTAL PEDIDO'] / df_comercial['TOTAL CUSTO']

# 2.Calculando o MKP REAL em % (Multiplicador real - 1) * 100 para comparar com o MKP do ERP
df_comercial['MKP REAL %'] = (df_comercial['MKP REAL'] - 1) * 100

# 3.Tratamento de dados em caso de erro
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].replace([float('inf'), float('-inf')], 0)  # Substitui inf e -inf por 0

# Preenchendo os valores nulos da coluna MKP com 0 (zero)
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].fillna(0)


## FALTA AJUSTAR AS COLUNAS DO MKP REAL PARA O FORMATO DE PORCENTAGEM (COM O SINAL DE %) E COM 2 CASAS DECIMAIS
## E TAMBÉM DELETAR A COLUNA DO MKP QUE ESTÁ ERRADA E DANDO VALORES INCORRETOS



KeyError: 'TOTAL ITENS'